In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Load dataset
df = pd.read_csv("cleaned_dataset.csv")
target = "Retard moyen de tous les trains à l'arrivée"
y = df[target]

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870.0,5.0,289.0,11.247809,3.693179,...,110.0,6.510000,44.0,8.000000,36.134454,31.092437,10.924370,15.966387,5.040000,0.840336
1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406.0,1.0,213.0,8.479969,4.567119,...,32.0,5.363539,9.0,4.000000,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226.0,0.0,21.0,6.239683,0.286283,...,11.0,2.938053,6.0,1.000000,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,71.0,7.235211,0.980000,...,39.0,5.292211,18.0,5.138432,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472.0,4.0,224.0,6.784673,3.229701,...,42.0,4.882372,10.0,0.000000,15.789474,45.614035,20.420203,15.789474,1.754386,1.754386


In [ ]:
# Date feature engineering
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["month"] = df["Date"].dt.month

features = [
    "Durée moyenne du trajet",
    "Nombre de circulations prévues",
    "Nombre de trains annulés",
    "month",
    "Nombre trains en retard > 15min",
    "Nombre trains en retard > 30min",
    "Nombre trains en retard > 60min",
    "Service"
]
X = df[features]

In [ ]:
# Encode categorical variable
X = pd.get_dummies(X, drop_first=True)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Optimized Random Forest
model = RandomForestRegressor(
    n_estimators=200,        
    max_depth=15,            
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\nRandom Forest Performance:")
print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

In [ ]:
# Regression Graph
plt.figure(figsize=(8,6))
plt.scatter(y_test, predictions, alpha=0.6)
plt.xlabel("Actual Delay")
plt.ylabel("Predicted Delay")
plt.title("Random Forest - Actual vs Predicted")

Baseline Performance:
RMSE: 4.012358701884818
MAE: 3.0261232803113467
R²: -0.0013137981060027393


In [ ]:
# Perfect prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red'
)
plt.show()

# Save lightweight model
joblib.dump(model, "linear_delay_model.pkl")
joblib.dump(X.columns, "model_features.pkl")

In [ ]:
# Perfect prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red'
)
plt.show()

# Feature importance
importances = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
print("\nTop 10 Most Important Features:")
print(importances.head(10))

# Save model
joblib.dump(model, "rf_delay_model.pkl")
joblib.dump(X.columns, "model_features.pkl")


Random Forest Performance:
RMSE: 1.4619054675433734
MAE: 1.0044908419493126
R²: 0.8670742014077208

Improvement over Baseline (RMSE reduction):
2.550453234341444


In [ ]:
y_pred = model.predict(X_test)

plt.figure(figsize=(16,10))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Real Delay")
plt.ylabel("Predicted Delay")
plt.title("Real vs Predicted")
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)
plt.show()